In [ ]:
# ==========================================
# CONFIGURAÇÃO DO CLIENTE VOYAGEAI
# ==========================================
# Carrega as variáveis de ambiente do arquivo .env (onde está sua API key)
# O serviço VoyageAI é usado para transformar texto em vetores numéricos (embeddings)

from dotenv import load_dotenv
import voyageai

load_dotenv()

# Cria um cliente para se conectar ao serviço VoyageAI
client = voyageai.Client()

c:\Projetos\Anthropic\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
# ==========================================
# FUNÇÃO DE CHUNKING (DIVISÃO DE TEXTO)
# ==========================================

import re

def chunk_by_section(document_text):
    """
    Divide um documento em pedaços menores (chunks) baseado nos títulos de seção.
    
    POR QUE FAZER ISSO?
    - Documentos grandes são difíceis de processar de uma vez
    - Dividir por seções mantém o contexto relacionado junto
    - Facilita encontrar a informação específica que o usuário procura
    
    COMO FUNCIONA:
    - Procura pelo padrão "\n## " (quebra de linha + título markdown nível 2)
    - Divide o texto toda vez que encontra esse padrão
    - Retorna uma lista com cada seção separada
    """
    pattern = r"\n## "
    return re.split(pattern, document_text)

In [ ]:
# ==========================================
# FUNÇÃO DE GERAÇÃO DE EMBEDDINGS
# ==========================================

def generate_embedding(chunks, model="voyage-3-large", input_type="query"):
    """
    Transforma texto em vetores numéricos (embeddings).
    
    O QUE SÃO EMBEDDINGS?
    - São representações numéricas do significado do texto
    - Textos com significados similares têm embeddings similares
    - Permitem que computadores "entendam" semelhança entre textos
    
    PARÂMETROS:
    - chunks: pode ser uma string única ou lista de strings
    - model: qual modelo de embedding usar (voyage-3-large é mais preciso)
    - input_type: "query" para perguntas, "document" para documentos
    
    COMO FUNCIONA:
    - Detecta se recebeu um texto único ou lista de textos
    - Chama a API do VoyageAI para gerar os embeddings
    - Retorna embedding único ou lista dependendo da entrada
    """
    is_list = isinstance(chunks, list)
    input = chunks if is_list else [chunks]
    result = client.embed(input, model=model, input_type=input_type)
    return result.embeddings if is_list else result.embeddings[0]

In [ ]:
# ==========================================
# CLASSE VECTORINDEX - BANCO DE DADOS DE VETORES
# ==========================================

import math
from typing import Optional, Any, List, Dict, Tuple


class VectorIndex:
    """
    Um índice de vetores simples para armazenar e buscar documentos.
    
    O QUE FAZ?
    - Armazena embeddings (vetores) e os documentos correspondentes
    - Permite buscar documentos similares a uma query
    - Calcula distâncias entre vetores para medir similaridade
    
    MÉTRICAS DE DISTÂNCIA:
    - "cosine": mede o ângulo entre vetores (mais comum em NLP)
    - "euclidean": mede a distância direta entre pontos
    """
    
    def __init__(
        self,
        distance_metric: str = "cosine",
        embedding_fn=None,
    ):
        """
        Inicializa o índice vazio.
        
        ATRIBUTOS:
        - vectors: lista que armazenará os embeddings
        - documents: lista que armazenará os documentos originais
        - _vector_dim: dimensão dos vetores (definida automaticamente)
        - _distance_metric: qual métrica usar para calcular similaridade
        - _embedding_fn: função opcional para gerar embeddings automaticamente
        """
        self.vectors: List[List[float]] = []
        self.documents: List[Dict[str, Any]] = []
        self._vector_dim: Optional[int] = None
        if distance_metric not in ["cosine", "euclidean"]:
            raise ValueError("distance_metric must be 'cosine' or 'euclidean'")
        self._distance_metric = distance_metric
        self._embedding_fn = embedding_fn

    def add_document(self, document: Dict[str, Any]):
        if not self._embedding_fn:
            raise ValueError(
                "Embedding function not provided during initialization."
            )
        if not isinstance(document, dict):
            raise TypeError("Document must be a dictionary.")
        if "content" not in document:
            raise ValueError(
                "Document dictionary must contain a 'content' key."
            )

        content = document["content"]
        if not isinstance(content, str):
            raise TypeError("Document 'content' must be a string.")

        vector = self._embedding_fn(content)
        self.add_vector(vector=vector, document=document)

    def search(
        self, query: Any, k: int = 1
    ) -> List[Tuple[Dict[str, Any], float]]:
        """
        Busca os k documentos mais similares à query.
        
        FLUXO DE BUSCA:
        1. Se query é string, gera embedding usando _embedding_fn
        2. Se query já é vetor, usa diretamente
        3. Calcula distância entre query e TODOS os vetores armazenados
        4. Ordena por distância (menor = mais similar)
        5. Retorna os k primeiros resultados
        
        RETORNA: Lista de tuplas (documento, distância)
        - Distância menor = mais similar
        """
        if not self.vectors:
            return []

        # Converte query em vetor se necessário
        if isinstance(query, str):
            if not self._embedding_fn:
                raise ValueError(
                    "Embedding function not provided for string query."
                )
            query_vector = self._embedding_fn(query)
        elif isinstance(query, list) and all(
            isinstance(x, (int, float)) for x in query
        ):
            query_vector = query
        else:
            raise TypeError(
                "Query must be either a string or a list of numbers."
            )

        if self._vector_dim is None:
            return []

        if len(query_vector) != self._vector_dim:
            raise ValueError(
                f"Query vector dimension mismatch. Expected {self._vector_dim}, got {len(query_vector)}"
            )

        if k <= 0:
            raise ValueError("k must be a positive integer.")

        # Seleciona função de distância apropriada
        if self._distance_metric == "cosine":
            dist_func = self._cosine_distance
        else:
            dist_func = self._euclidean_distance

        # Calcula distância para cada vetor armazenado
        distances = []
        for i, stored_vector in enumerate(self.vectors):
            distance = dist_func(query_vector, stored_vector)
            distances.append((distance, self.documents[i]))

        # Ordena por distância (menor primeiro = mais similar)
        distances.sort(key=lambda item: item[0])

        # Retorna os k melhores resultados
        return [(doc, dist) for dist, doc in distances[:k]]

    def add_vector(self, vector, document: Dict[str, Any]):
        """
        Adiciona um vetor e seu documento ao índice.
        
        VALIDAÇÕES IMPORTANTES:
        - Verifica se o vetor é uma lista de números
        - Verifica se o documento tem a chave 'content'
        - Garante que todos os vetores têm a mesma dimensão
        
        POR QUE A MESMA DIMENSÃO?
        - Só podemos comparar vetores de mesmo tamanho
        - É como comparar coordenadas: (x,y) com (x,y,z) não faz sentido
        """
        if not isinstance(vector, list) or not all(
            isinstance(x, (int, float)) for x in vector
        ):
            raise TypeError("Vector must be a list of numbers.")
        if not isinstance(document, dict):
            raise TypeError("Document must be a dictionary.")
        if "content" not in document:
            raise ValueError(
                "Document dictionary must contain a 'content' key."
            )

        # Define a dimensão no primeiro vetor adicionado
        if not self.vectors:
            self._vector_dim = len(vector)
        elif len(vector) != self._vector_dim:
            raise ValueError(
                f"Inconsistent vector dimension. Expected {self._vector_dim}, got {len(vector)}"
            )

        self.vectors.append(list(vector))
        self.documents.append(document)

    def _euclidean_distance(
        self, vec1: List[float], vec2: List[float]
    ) -> float:
        """
        Calcula distância euclidiana (linha reta) entre dois vetores.
        
        FÓRMULA: √(Σ(p-q)²)
        - É como medir a distância entre dois pontos no espaço
        - Exemplo 2D: distância entre (1,2) e (4,6) = √((4-1)² + (6-2)²) = 5
        """
        if len(vec1) != len(vec2):
            raise ValueError("Vectors must have the same dimension")
        return math.sqrt(sum((p - q) ** 2 for p, q in zip(vec1, vec2)))

    def _dot_product(self, vec1: List[float], vec2: List[float]) -> float:
        """
        Calcula produto escalar entre dois vetores.
        
        FÓRMULA: Σ(p × q)
        - Multiplica elementos correspondentes e soma tudo
        - Usado no cálculo de similaridade de cosseno
        """
        if len(vec1) != len(vec2):
            raise ValueError("Vectors must have the same dimension")
        return sum(p * q for p, q in zip(vec1, vec2))

    def _magnitude(self, vec: List[float]) -> float:
        """
        Calcula magnitude (tamanho/comprimento) de um vetor.
        
        FÓRMULA: √(Σx²)
        - É como medir o comprimento de uma flecha do ponto zero até o ponto
        """
        return math.sqrt(sum(x * x for x in vec))

    def _cosine_distance(self, vec1: List[float], vec2: List[float]) -> float:
        """
        Calcula distância de cosseno entre dois vetores.
        
        POR QUE USAR COSSENO?
        - Mede o ângulo entre vetores, não a distância absoluta
        - Vetores apontando na mesma direção são similares
        - Funciona bem para comparar textos independente do tamanho
        
        FÓRMULA:
        1. Similaridade de cosseno = (vec1 · vec2) / (|vec1| × |vec2|)
        2. Distância de cosseno = 1 - similaridade de cosseno
        
        RESULTADO: 0 = idênticos, 1 = completamente diferentes
        """
        if len(vec1) != len(vec2):
            raise ValueError("Vectors must have the same dimension")

        mag1 = self._magnitude(vec1)
        mag2 = self._magnitude(vec2)

        # Casos especiais: vetores zerados
        if mag1 == 0 and mag2 == 0:
            return 0.0
        elif mag1 == 0 or mag2 == 0:
            return 1.0

        dot_prod = self._dot_product(vec1, vec2)
        cosine_similarity = dot_prod / (mag1 * mag2)
        # Garante que o valor está entre -1 e 1 (por precisão numérica)
        cosine_similarity = max(-1.0, min(1.0, cosine_similarity))

        # Converte similaridade em distância
        return 1.0 - cosine_similarity

    def __len__(self) -> int:
        return len(self.vectors)

    def __repr__(self) -> str:
        has_embed_fn = "Yes" if self._embedding_fn else "No"
        return f"VectorIndex(count={len(self)}, dim={self._vector_dim}, metric='{self._distance_metric}', has_embedding_fn='{has_embed_fn}')"

In [ ]:
# ==========================================
# PASSO 1: CARREGAR O DOCUMENTO
# ==========================================
# Lê o arquivo report.md que contém o texto a ser indexado

with open("./report.md", "r") as f:
    text = f.read()

In [ ]:
# ==========================================
# PASSO 2: DIVIDIR EM CHUNKS
# ==========================================
# Divide o documento em seções menores para processamento
# Cada seção começa com "## " (título markdown nível 2)

chunk = chunk_by_section(text)  

# Visualiza o 4º chunk como exemplo
chunk[3]

'Methodology\n\nThe insights compiled within this Annual Interdisciplinary Research Review represent a synthesis of findings drawn from standard departmental reporting cycles, specialized project updates, and cross-functional review meetings conducted throughout the year. Data sources included internal project databases, laboratory notebooks, financial reporting systems, legal case summaries, security incident logs, and minutes from dedicated working groups. A central review committee, comprising representatives nominated by each division head, was tasked with identifying key developments and potential cross-domain implications. This committee utilized a standardized reporting template to capture essential details, including unique identifiers (project codes, error numbers, case references, etc.) and progress metrics. Subsequent analysis focused on identifying thematic overlaps, shared challenges, and opportunities for synergistic development, forming the basis of this consolidated rep

In [ ]:
# ==========================================
# PASSO 3: GERAR EMBEDDINGS PARA CADA CHUNK
# ==========================================
# Transforma cada pedaço de texto em um vetor numérico
# Todos os chunks são processados de uma vez (bulk operation)
# Isso evita problemas de rate limiting da API

chunk_embeddings = generate_embedding(chunk)

In [ ]:
# ==========================================
# PASSO 4: CRIAR O ÍNDICE E POPULAR COM OS DADOS
# ==========================================
# Armazena cada embedding junto com seu texto original no índice
# O índice mantém a relação entre vetor e documento

store = VectorIndex()
for chunk_embedding, chunk_text in zip(chunk_embeddings, chunk):
    store.add_vector(vector=chunk_embedding, document={"content": chunk_text})

In [ ]:
# ==========================================
# PASSO 5: PROCESSAR UMA PERGUNTA DO USUÁRIO
# ==========================================
# Quando um usuário faz uma pergunta, geramos um embedding para ela também
# Isso nos permite comparar a pergunta com os documentos armazenados

user_embedding = generate_embedding("What did the software engineering dept do last year ?")

In [ ]:
# ==========================================
# PASSO 6: BUSCAR E EXIBIR RESULTADOS RELEVANTES
# ==========================================
# Busca os 2 chunks mais similares à pergunta do usuário
# O sistema compara o embedding da pergunta com todos os embeddings armazenados
# e retorna os documentos com menor distância (mais similares)

results = store.search(user_embedding, k=2)

for doc, dist in results:
    print(f"Distance: {dist:.4f}")  # Menor distância = mais relevante
    print(f"Content: {doc['content'][:200]}...")  # Mostra os primeiros 200 caracteres
    print("-" * 50)

Distance: 0.4996
Content: Section 2: Software Engineering - Project Phoenix Stability Enhancements

The Software Engineering division dedicated considerable effort to improving the stability and performance of the core systems...
--------------------------------------------------
Distance: 0.5054
Content: Future Directions

This year's cross-domain insights underscore the interconnectedness of our diverse research and operational activities. The stability enhancements achieved in Software Engineering (...
--------------------------------------------------
